In [362]:
import pandas as pd
from operator import itemgetter
import networkx as nx
import numpy as np

In [363]:
pre_data = pd.read_csv('data/pre_survey.csv')
post_data = pd.read_csv('data/post_survey.csv')

### Nodes

In [364]:
pre_nodes = pre_data[['ID','Name']] #only need these columns, we fit everything to name for ease (only 38 people)
pre_nodes = pre_nodes.rename(columns={'Name':'Label'})
print("There are", len(pre_nodes), "nodes.")

There are 38 nodes.


In [365]:
pre_nodes.to_csv('data/pre_nodes.csv', index=False)

In [366]:
post_nodes = post_data[['ID','Name']] #only need these columns, we fit everything to name for ease (only 38 people)
post_nodes = post_nodes.rename(columns={'Name':'Label'})
print("There are", len(post_nodes), "nodes.")

There are 38 nodes.


In [367]:
post_nodes.to_csv('data/post_nodes.csv', index=False)

### Edges

In [368]:
working = pre_data.iloc[:,4:] #removes this column
working = working.set_index('Name').drop(columns=['NetID']) #set index for stacking, remove other cols
working = working.stack().rename_axis(['Source','Target']).reset_index() #makes into a stack from matrix
pre_edges = working.rename(columns={0:'Weight'}) #rename columns
pre_edges = pre_edges[pre_edges['Target'].isin(pre_nodes['Label'])] #only include targets in the club
pre_edges.head()

,Source,Target,Weight
0,Nikhil Chinchalkar,Nikhil Chinchalkar,I am this person
2,Nikhil Chinchalkar,Rithya Sriram,I speak with them at least once a week
3,Nikhil Chinchalkar,Carina Lau,I speak with them at least once a week
4,Nikhil Chinchalkar,Jenny Williams,I speak with them at least once a week
5,Nikhil Chinchalkar,Rahi Dasgupta,I speak with them at least once a week


In [369]:
len(pre_edges['Source'].unique()), len(pre_edges['Target'].unique()) #sanity check, should be 38 people still

(38, 38)

In [370]:
pre_edges['Weight'].unique() #used for mapping weights below

array(['I am this person', 'I speak with them at least once a week',
       "I've spoken to them more than once before",
       'I recognize their face/name', "I've spoken to them once before",
       "I've never seen/heard of this person before",
       'I speak with them everyday'], dtype=object)

In [371]:
weights_map = {'I am this person':0,
               'I speak with them everyday':4,
               'I speak with them at least once a week':3,
               "I've spoken to them more than once before":2,
               "I've spoken to them once before":1,
               'I recognize their face/name':0,
               "I've never seen/heard of this person before":0} #subject to change

In [372]:
pre_edges['Weight'] = pre_edges['Weight'].map(lambda x: weights_map[x])

In [373]:
pre_edges.to_csv('data/pre_edges.csv', index=False)

In [374]:
working = post_data.iloc[:,4:] #removes this column
working = working.set_index('Name').drop(columns=['NetID']) #set index for stacking, remove other cols
working = working.stack().rename_axis(['Source','Target']).reset_index() #makes into a stack from matrix
post_edges = working.rename(columns={0:'Weight'}) #rename columns
post_edges = post_edges[post_edges['Target'].isin(post_nodes['Label'])] #only include targets in the club
print(len(post_edges['Source'].unique()), len(post_edges['Target'].unique())) #sanity check, should be 38 people still
post_edges['Weight'] = post_edges['Weight'].map(lambda x: weights_map[x])
post_edges.to_csv('data/post_edges.csv', index=False)

38 38


### Demographics

In [375]:
demographics = pd.read_csv('data/demographics.csv')
pre_demographics = pd.merge(demographics, pre_nodes, left_on='Full Name', right_on='Label', how='right')
pre_demographics.to_csv('data/pre_node_demographics.csv', index=False)

In [376]:
demographics = pd.read_csv('data/demographics.csv')
post_demographics = pd.merge(demographics, post_nodes, left_on='Full Name', right_on='Label', how='right')
post_demographics.to_csv('data/post_node_demographics.csv', index=False)

### Graph

In [377]:
pre_undirected_edges = pd.merge(pre_edges, pre_edges, left_on=['Target','Source'], right_on=['Source','Target'], how='left')
pre_undirected_edges['Average Weight'] = np.nanmean(pre_undirected_edges[['Weight_x','Weight_y']], axis=1)
pre_undirected_edges = pre_undirected_edges.rename(columns={'Source_x':'Source'})
pre_undirected_edges = pre_undirected_edges.rename(columns={'Target_x':'Target'})
pre_undirected_edges = pre_undirected_edges[['Source', 'Target','Average Weight']]

In [378]:
post_undirected_edges = pd.merge(post_edges, post_edges, left_on=['Target','Source'], right_on=['Source','Target'], how='left')
post_undirected_edges['Average Weight'] = np.nanmean(post_undirected_edges[['Weight_x','Weight_y']], axis=1)
post_undirected_edges = post_undirected_edges.rename(columns={'Source_x':'Source'})
post_undirected_edges = post_undirected_edges.rename(columns={'Target_x':'Target'})
post_undirected_edges = post_undirected_edges[['Source', 'Target','Average Weight']]

In [379]:
pre_undirected_edges = pd.concat([pd.DataFrame(np.sort(pre_undirected_edges[['Source','Target']], axis=1), columns=['Source','Target']), 
           pre_undirected_edges['Average Weight']], axis=1).drop_duplicates() #gets rid of duplicate edges, since undirected

post_undirected_edges = pd.concat([pd.DataFrame(np.sort(post_undirected_edges[['Source','Target']], axis=1), columns=['Source','Target']), 
           post_undirected_edges['Average Weight']], axis=1).drop_duplicates() #gets rid of duplicate edges, since undirected

In [380]:
pre_undirected_edges = pre_undirected_edges[pre_undirected_edges['Average Weight'] != 0]
pre_undirected_edges = pre_undirected_edges.rename(columns={'(Source,)':'Source'})
pre_undirected_edges = pre_undirected_edges.rename(columns={'(Target,)':'Target'})

In [381]:
post_undirected_edges = post_undirected_edges[post_undirected_edges['Average Weight'] != 0]
post_undirected_edges = post_undirected_edges.rename(columns={'(Source,)':'Source'})
post_undirected_edges = post_undirected_edges.rename(columns={'(Target,)':'Target'})

In [382]:
pre_G_undirected = nx.from_pandas_edgelist(
    pre_undirected_edges,
    source='Source',
    target='Target',
    edge_attr=['Average Weight'])

In [383]:
post_G_undirected = nx.from_pandas_edgelist(
    post_undirected_edges,
    source='Source',
    target='Target',
    edge_attr=['Average Weight'])

In [384]:
for _, row in demographics.iterrows():
    node_id = row['Full Name']

    pre_G_undirected.add_node(node_id)
    for col in demographics.columns:
        if col != 'Full Name':
            pre_G_undirected.nodes[node_id][col] = row[col]
            
    post_G_undirected.add_node(node_id)
    for col in demographics.columns:
        if col != 'Full Name':
            post_G_undirected.nodes[node_id][col] = row[col]

In [385]:
nx.write_gexf(pre_G_undirected, "pre_cdj_network.gexf")
nx.write_gexf(post_G_undirected, "post_cdj_network.gexf")

### Metrics

In [416]:
degrees = [degree for name, degree in pre_G_undirected.degree()] #can include weight='Average Weight' in degree() too
print("Pre-network avg. degree:", np.average(degrees))
degrees = [degree for name, degree in post_G_undirected.degree()] #can include weight='Average Weight' in degree() too
print("Post-network avg. degree:", np.average(degrees))

Pre-network avg. degree: 9.473684210526315
Post-network avg. degree: 15.0


In [386]:
density = nx.density(pre_G_undirected)
print("Pre-network density:", density)
density = nx.density(post_G_undirected)
print("Post-network density:", density)

Pre-network density: 0.25604551920341395
Post-network density: 0.40540540540540543


In [387]:
print(nx.is_connected(pre_G_undirected))
print(nx.is_connected(post_G_undirected))

True
True


In [388]:
diameter = nx.diameter(pre_G_undirected)
print("Post network diameter:", diameter)
diameter = nx.diameter(post_G_undirected)
print("Post network diameter:", diameter)

Post network diameter: 3
Post network diameter: 2


In [389]:
pair_dict = dict(nx.all_pairs_all_shortest_paths(pre_G_undirected, weight=None))
all_paths = []
for person in pair_dict.keys():
    for target in pair_dict[person]:
        for pair in pair_dict[person][target]:
            all_paths.append(pair)

In [390]:
triadic_closure = nx.transitivity(pre_G_undirected)
print("Pre triadic closure:", triadic_closure)
triadic_closure = nx.transitivity(post_G_undirected)
print("Post triadic closure:", triadic_closure)

Pre triadic closure: 0.4991883116883117
Post triadic closure: 0.5526471167093092


In [391]:
degree_dict = dict(pre_G_undirected.degree(pre_G_undirected.nodes(), weight='Average Weight'))
sorted_degree = sorted(degree_dict.items(), key=itemgetter(1), reverse=True)[:5]
print("Pre-degrees:",sorted_degree)
degree_dict = dict(post_G_undirected.degree(post_G_undirected.nodes(), weight='Average Weight'))
sorted_degree = sorted(degree_dict.items(), key=itemgetter(1), reverse=True)[:5]
print("Post-degrees:",sorted_degree)

Pre-degrees: [('Nikhil Chinchalkar', 49.5), ('Rithya Sriram', 42.5), ('Mei Knight', 35.5), ('Rahi Dasgupta', 33.0), ('Jenny Williams', 31.0)]
Post-degrees: [('Nikhil Chinchalkar', 81.0), ('Rithya Sriram', 61.0), ('Jenny Williams', 60.0), ('Mei Knight', 45.0), ('Tianyi Chen', 44.5)]


In [ ]:
eigenvector_dict = nx.eigenvector_centrality(pre_G_undirected, weight='Average Weight') # Run eigenvector centrality
sorted_eigenvector = sorted(eigenvector_dict.items(), key=itemgetter(1), reverse=True)
sorted_eigenvector[:10]

[('Rithya Sriram', 0.3653358825273493),
 ('Nikhil Chinchalkar', 0.3581673502347927),
 ('Jenny Williams', 0.28800651458464954),
 ('Mei Knight', 0.28448615231457774),
 ('Rahi Dasgupta', 0.2652811981676298),
 ('Tianyi Chen', 0.25546200272851677),
 ('Carina Lau', 0.2509785183011594),
 ('Natalie Miller', 0.23266329802743385),
 ('Vivian Guo', 0.23160052611947154),
 ('Arjun Maitra', 0.22417561868831382)]

In [401]:
eigenvector_dict = nx.eigenvector_centrality(post_G_undirected, weight='Average Weight') # Run eigenvector centrality
sorted_eigenvector = sorted(eigenvector_dict.items(), key=itemgetter(1), reverse=True)
sorted_eigenvector[:10]

[('Nikhil Chinchalkar', 0.36072387486286805),
 ('Rithya Sriram', 0.3084890524195617),
 ('Jenny Williams', 0.2949397287687682),
 ('Mei Knight', 0.2530557174825698),
 ('Tianyi Chen', 0.24105121545451677),
 ('Rahi Dasgupta', 0.22980938726799469),
 ('Vivian Guo', 0.22122001652123313),
 ('Natalie Miller', 0.20725313767891138),
 ('Kayla Amkraut', 0.19888443020115198),
 ('Carina Lau', 0.1863349559130768)]